# Phase 4 — LLM Agents & Forecasters

1. **TradingAgents** full crew via `aqp.agents.trading.propagate`: analysts (fundamentals / sentiment / news / technical) → bull/bear debate → trader → risk manager → portfolio manager.
2. **FinGPT-Forecaster** as an `IAlphaModel` (`ForecasterAlpha`).
3. **FinBERT / FinGPT sentiment** scorer over a news feed.
4. **FinRobot role packs** — `MarketForecaster`, `DocumentAnalyzer`, `FinancialReportBuilder`, `TradeStrategist`, and the umbrella `EquityResearch`.
5. **LLM Screener** — `LLMScreenerAlpha` + `LLMUniverseSelector` on a candidate universe.

Deps: `pip install "agentic-quant-platform[llm-finance]"` and a reachable LLM (Ollama at `http://localhost:11434` by default).

## 1. TradingAgents full crew

In [ ]:
from aqp.agents.trading.propagate import propagate

decision = propagate(
    vt_symbol='NVDA.NASDAQ',
    as_of='2024-10-15',
    max_debate_rounds=1,  # keep the demo cheap
)
decision

## 2. FinGPT-Forecaster alpha

In [ ]:
from datetime import datetime
from aqp.core.types import Symbol
from aqp.ml.applications.forecaster.alpha import ForecasterAlpha

alpha = ForecasterAlpha(n_past_weeks=2, min_confidence=0.4, strength=0.15)
signals = alpha.generate_signals(
    bars=None,  # alpha will fetch news internally
    universe=[Symbol(ticker='AAPL'), Symbol(ticker='MSFT')],
    context={'current_time': datetime(2024, 10, 15)},
)
[s.__dict__ for s in signals]

## 3. Sentiment scorer

In [ ]:
from aqp.ml.applications.sentiment import get_scorer

scorer = get_scorer()  # defaults to yiyanghkust/finbert-tone
headlines = [
    'NVDA earnings beat; guidance strong',
    'Regulatory probe launched into Apple App Store fees',
    'Tesla recalls 500k vehicles over braking defect',
]
list(zip(headlines, scorer.score(headlines)))

## 4. FinRobot — EquityResearch umbrella crew

In [ ]:
from aqp.agents.financial import EquityResearch

crew = EquityResearch(tier='deep')
report = crew.run(
    ticker='NVDA',
    as_of='2024-10-15',
    fundamentals={'pe': 62, 'rev_growth': 0.82},
    technicals={'rsi_14': 58, 'macd_hist': 1.2},
    sentiment={'mean_score_7d': 0.65},
    price_summary={'close': 132.9, 'return_1m': 0.11},
    news_digest=[{'title': 'NVDA wins Saudi deal', 'sentiment': 0.8}],
    documents=[
        ('10-Q Q3 2024', 'What are the key risk factors flagged this quarter?', 
         'Risk factors include supply-chain concentration in TSMC, export controls on advanced AI chips, and customer concentration with hyperscalers...'),
    ],
)
print(report.to_json())

## 5. LLM Screener

In [ ]:
from aqp.agents.screening import LLMScreener

screener = LLMScreener(k=5, tier='deep', min_conviction=0.5)
snapshots = [
    {'ticker': 'NVDA', 'ret_5d': 0.06, 'ret_20d': 0.11, 'vol': 0.03, 'pe': 62},
    {'ticker': 'AAPL', 'ret_5d': -0.01, 'ret_20d': 0.02, 'vol': 0.015, 'pe': 32},
    {'ticker': 'TSLA', 'ret_5d': -0.04, 'ret_20d': -0.08, 'vol': 0.05, 'pe': 70},
    {'ticker': 'MSFT', 'ret_5d': 0.02, 'ret_20d': 0.05, 'vol': 0.014, 'pe': 35},
    {'ticker': 'AMD',  'ret_5d': 0.03, 'ret_20d': 0.09, 'vol': 0.04, 'pe': 45},
]
screener.rank(snapshots, as_of='2024-10-15')